In [0]:
# Storage configuration
storage_account_name = "stkalshianalytics"
storage_account_key = dbutils.secrets.get("kalshi-scope", "adls-storage-key")

spark.conf.set(
    f"fs.azure.account.key.{storage_account_name}.dfs.core.windows.net",
    storage_account_key
)

bronze_path = f"abfss://bronze@{storage_account_name}.dfs.core.windows.net"
silver_path = f"abfss://silver@{storage_account_name}.dfs.core.windows.net"
gold_path = f"abfss://gold@{storage_account_name}.dfs.core.windows.net"

print("Storage configured successfully")
print(f"Bronze: {bronze_path}")
print(f"Silver: {silver_path}")
print(f"Gold: {gold_path}")

In [0]:
# Verify bronze data is readable
from datetime import datetime

today = datetime.utcnow().strftime("%Y/%m/%d")
bronze_file = f"{bronze_path}/{today}/kalshi_markets.json"

df_raw = spark.read.option("multiline", "true").json(bronze_file)
print(f"Bronze file loaded successfully")
print(f"Schema:")
df_raw.printSchema()

In [0]:
from pyspark.sql.functions import explode, col

df_markets = df_raw.select(
    col("ingestion_timestamp"),
    explode(col("markets")).alias("market")
)

df_flat = df_markets.select(
    col("ingestion_timestamp"),
    col("market.ticker"),
    col("market.title"),
    col("market.event_ticker"),
    col("market.status"),
    col("market.market_type"),
    col("market.yes_ask_dollars").cast("double").alias("yes_ask"),
    col("market.yes_bid_dollars").cast("double").alias("yes_bid"),
    col("market.no_ask_dollars").cast("double").alias("no_ask"),
    col("market.no_bid_dollars").cast("double").alias("no_bid"),
    col("market.last_price_dollars").cast("double").alias("last_price"),
    col("market.liquidity_dollars").cast("double").alias("liquidity"),
    col("market.volume_24h_fp").cast("double").alias("volume_24h"),
    col("market.open_interest_fp").cast("double").alias("open_interest"),
    col("market.close_time").cast("timestamp").alias("close_time"),
    col("market.open_time").cast("timestamp").alias("open_time"),
    col("market.is_provisional"),
    col("market.mve_collection_ticker")
)

print(f"Total markets: {df_flat.count()}")
df_flat.show(5, truncate=False)

In [0]:
# Show a cleaner sample of single-leg markets with actual liquidity
df_liquid = df_flat.filter(
    (col("liquidity") > 0) & 
    (col("mve_collection_ticker").isNull())
)

print(f"Single markets with liquidity: {df_liquid.count()}")

df_liquid.select(
    "ticker",
    "title",
    "yes_ask",
    "yes_bid",
    "liquidity",
    "volume_24h",
    "close_time"
).show(10, truncate=50)

In [0]:
from datetime import datetime, timezone, timedelta

storage_account_name = "stkalshianalytics"
storage_account_key = dbutils.secrets.get("kalshi-scope", "adls-storage-key")

spark.conf.set(
    f"fs.azure.account.key.{storage_account_name}.dfs.core.windows.net",
    storage_account_key
)

bronze_path = f"abfss://bronze@{storage_account_name}.dfs.core.windows.net"

yesterday = (datetime.now(timezone.utc) - timedelta(days=1)).strftime("%Y/%m/%d")
bronze_file = f"{bronze_path}/{yesterday}/kalshi_markets.json"

df_raw = spark.read.option("multiline", "true").json(bronze_file)
df_raw.select("market_count").show()